In [0]:
%run ../../config/utils

In [0]:
import argparse
from datetime import datetime

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
import yaml

import sys
sys.path.append('..')
sys.path.append('../..')

import lib_trip_spend.python_general_utilities as util_func
import mlflow
from mlflow.models.signature import infer_signature
from mlflow.client import MlflowClient
import matplotlib.pyplot as plt
import shap
import numpy as np
mlflow.autolog(disable=True)

In [0]:
############## SET VARIABLES FROM CONFIG ###########################
with open('./config/config.yml', "r") as stream:
    config = yaml.load(stream, Loader=yaml.FullLoader)

LAST_FISCAL_WEEK_TRAINING = config["etl"]["end_date"]
FIRST_FISCAL_WEEK_TRAINING = config["etl"]["start_date"]
START_WINDOW = config["spend"]["start_window"] - 1
END_WINDOW = config["spend"]["end_window"] - 1
SPLIT_RATE = config["shared"]["split_rate"]
SAMPLE_RATE = config["shared"]["sample_rate"]
BUCKET = config["shared"]["bucket"]
NUMBER_OF_TREES = config["trip"]["number_of_trees"]
MAX_DEPTH = config["trip"]["max_depth"]
MAX_FEATURES = config["trip"]["max_features"]
MIN_LEAF_SIZE = config["trip"]["min_sample_leaf"]
MODEL = config["trip"]["model"]
DATE = datetime.today().strftime("%Y%m%d")
DATASET_PATH = config["shared"]["dataset_path"]
PATH_PREFIX = config["shared"]["base_path"]
MODEL_PATH_PREFIX = config["shared"]["model_path"]
INPUT_DATA_PATH = "{}/{}/{}/{}".format(
    config["shared"]["base_path"],
    config["shared"]["dataset_path"],
    config["shared"]["run_name"],
    config["shared"]["etl_path"],
)

ETL_OUTPUT_PATH = "{}/{}/{}/{}-{}/transformed_customer_data".format(
    config["shared"]["base_path"],
    config["shared"]["dataset_path"],
    config["shared"]["run_name"],
    LAST_FISCAL_WEEK_TRAINING,
    FIRST_FISCAL_WEEK_TRAINING,
)

# FEATURE_PATH = config["shared"]["feature_path"]
REGRESSION_METRICS = config["spend"]["metrics"]
RUN_NAME = config["shared"]["run_name"]


In [0]:
############## DEFINE PATHS #######################################
OUTPUT_PATH = "s3://{}/{}/{}/{}/".format(
    BUCKET,
    config["shared"]["base_path"],
    config["shared"]["model_path"],
    RUN_NAME,
)

FEATURE_IMP_PATH = "%s%s_spend_V1_%s_features_%s_%s.csv" % (
    OUTPUT_PATH,
    MODEL,
    DATE,
    START_WINDOW,
    START_WINDOW + 2,
)

METRIC_PATH = "%s%s_spend_V1_%s_metric_%s_%s.csv" % (
    OUTPUT_PATH,
    MODEL,
    DATE,
    START_WINDOW,
    START_WINDOW + 2,
)
MODEL_PATH = "%s%s_spend_V1_%s.pkl" % (OUTPUT_PATH, MODEL, DATE)

In [0]:
############# DEFINE FUNCTIONS########################################
def create_rf_model(
    training_X,
    training_y,
    testing_X,
    max_depth,
    max_features,
    n_estimators,
    min_samples_leaf,
    features,
):
    model = RandomForestRegressor(
        n_jobs=-1,
        max_depth=max_depth,
        max_features=max_features,
        warm_start=True,
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
    )

    training_X.reset_index(inplace=True)
    testing_X.reset_index(inplace=True)
    training_X = training_X[features]
    testing_X = testing_X[features]
    model.fit(training_X, training_y.values.ravel())
    return model, model.predict(testing_X), model.feature_importances_


def create_lr_model(training_X, training_y, testing_X):
    training_X = training_X[training_X.columns.intersection(features)]
    testing_X = testing_X[training_X.columns.intersection(features)]
    model = ElasticNet(l1_ratio=1, max_iter=1000, warm_start=True)
    model.fit(training_X, training_y.values.ravel())
    return model, model.predict(testing_X), model.coef_

In [0]:
features = pd.read_csv(feature_path)

label_column = "spend_from_%s_%s" % (str(START_WINDOW), str(END_WINDOW))
trip_column = "will_visit_from_5_7"
(
    column_headers,
    categorical_features,
    continious_features,
) = util_func.get_column_headers(features, [label_column])
feature_headers = [x for x in column_headers if x != label_column]

In [0]:
data = spark.table(model_trip_spend_etl_output).select(*(column_headers + ["will_visit_from_5_7"])) 


In [0]:
training, testing = util_func.test_and_train_to_pandas(
    data.toPandas(), SPLIT_RATE, SAMPLE_RATE
)  # change back to sample rate

training, testing = util_func.train_initial_model(
    training.drop(columns=[trip_column, label_column]),
    training[[trip_column]],
    testing.drop(columns=[trip_column, label_column]),
    SPLIT_RATE,
    testing[[trip_column]],
    label_column,
)

In [0]:
experiment_name = experiment_name_trip_spend

mlflow.sklearn.autolog(disable=False, log_input_examples=True, log_models=False, log_datasets=False)

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri('databricks-uc')

if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name) 
mlflow.set_experiment(experiment_name)

feature_dataset = mlflow.data.from_spark(data, name = 'trips_spend_etl_dataset')

In [0]:
%sh
mkdir tmp_spend

In [0]:
mark_datetime = datetime.strftime(datetime.now(), '_%Y-%m-%d_%H-%M-%S')
with mlflow.start_run(run_name=f'training_{RUN_NAME}_{mark_datetime}') as run:
    mlflow.log_input(feature_dataset, context="source")
    mlflow.log_input(mlflow.data.from_pandas(training, source=feature_dataset.source), context="training")
    mlflow.log_input(mlflow.data.from_pandas(testing, source=feature_dataset.source), context="testing")

    if MODEL == "rf":
        model, predictions, feature_importance = create_rf_model(
            training.drop(columns=[label_column]),
            training[[label_column]],
            testing.drop(columns=[label_column]),
            MAX_DEPTH,
            MAX_FEATURES,
            NUMBER_OF_TREES,
            MIN_LEAF_SIZE,
            feature_headers,
        )
    if MODEL == "lr":
        model, predictions, feature_importance = create_lr_model(
            training.drop(columns=[label_column]),
            training[[label_column]],
            testing.drop(columns=[label_column]),
        )
    model_metrics = [
        util_func.create_spend_propensity_metric(
            predictions, testing[[label_column]]
        )
    ]
    model_metrics = util_func.get_df_from_list(model_metrics, REGRESSION_METRICS)
    feature_importances_pd = pd.DataFrame(
            util_func.create_sklearn_features(feature_importance, column_headers)
        )
    
    model_metrics.to_csv(spend_metrics_path, index=False)
    feature_importances_pd.to_csv(spend_feature_importance_path, index=False)
    mlflow.log_artifact(spend_metrics_path)
    mlflow.log_artifact(spend_feature_importance_path)

    signature = infer_signature(training.drop(columns=[label_column]), training[[label_column]])

    model_info = mlflow.sklearn.log_model(
            sk_model = model,
            artifact_path = "model",
            signature = signature,
            registered_model_name = trip_spend_model_catalog
        )



    feature_cols = list(getattr(model, "feature_names_in_", 
                                training.drop(columns=[label_column]).columns))

    X_eval = testing[feature_cols].sample(n=min(5000, len(testing)), random_state=42)

    bg_size = min(200, len(training))
    X_bg = training[feature_cols].sample(n=bg_size, random_state=42)

    if MODEL == "rf":
        explainer = shap.TreeExplainer(model)          
        sv = explainer.shap_values(X_eval)    

    elif MODEL == "lr":
        explainer = shap.LinearExplainer(
            model, X_bg, feature_perturbation="interventional"
        )
        sv = explainer.shap_values(X_eval)   

    plt.figure(figsize=(10, 8))
    shap.summary_plot(sv, X_eval, show=False)
    plt.tight_layout()
    plt.savefig("./tmp_spend/shap_summary_plot.png"); plt.close()

    plt.figure(figsize=(10, 8))
    shap.summary_plot(sv, X_eval, plot_type="bar", show=False)
    plt.tight_layout()
    plt.savefig("./tmp_spend/shap_feature_importance_plot.png"); plt.close()

    mlflow.log_artifact("./tmp_spend/shap_summary_plot.png", artifact_path="shap_plots")
    mlflow.log_artifact("./tmp_spend/shap_feature_importance_plot.png", artifact_path="shap_plots")


In [0]:
client = MlflowClient()
client.set_registered_model_alias(trip_spend_model_catalog, "champion", model_info.registered_model_version)

In [0]:
%sh
rm -r tmp_spend